In [1]:
%pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 13.1 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]2m1/2 [gensim]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import gensim.downloader as api
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load models (Note: These are large, start with English for the proof of concept)
model_en = api.load("glove-wiki-gigaword-100") 
# For your full project, you'd add:
# model_es = api.load("fasttext-wiki-news-subwords-300-es")
# model_ru = api.load("fasttext-wiki-news-subwords-300-ru")

[==================================================] 100.0% 128.1/128.1MB downloaded


In [3]:
def get_gender_direction(model, male_words, female_words):
    differences = []
    for m, f in zip(male_words, female_words):
        if m in model and f in model:
            differences.append(model[m] - model[f])
    return np.mean(differences, axis=0)

# English Anchors
en_m = ['man', 'male', 'boy', 'father']
en_f = ['woman', 'female', 'girl', 'mother']
gender_vector_en = get_gender_direction(model_en, en_m, en_f)

# Spanish Anchors (Note the grammatical markers)
es_m = ['hombre', 'varón', 'niño', 'padre']
es_f = ['mujer', 'hembra', 'niña', 'madre']

# Russian Anchors (Using Nominative Singular)
ru_m = ['мужчина', 'мужской', 'мальчик', 'отец']
ru_f = ['женщина', 'женский', 'девочка', 'мать']

In [30]:
def calculate_bias(model, target_word, gender_vector):
    if target_word in model:
        target_vec = model[target_word]
        # Cosine similarity between the word and the gender 'axis'
        bias_score = cosine_similarity(
            target_vec.reshape(1, -1), 
            gender_vector.reshape(1, -1)
        )[0][0]
        return bias_score
    return None

# Example test:
score = calculate_bias(model_en, "doctor", gender_vector_en)

def pretty_print_scores_en(word_list, label=''):
    if label:
        label += ': '
    print(label + '; '.join([f'{word}: {float(calculate_bias(model_en, word, gender_vector_en))}' for word in word_list]))

print('===== baseline words =====')
pretty_print_scores_en(en_m, 'M')
pretty_print_scores_en(en_f, 'F')
print('===== test words =====')
pretty_print_scores_en(['leader', 'authority', 'president', 'king', 'queen', 'premier', 'emperor', 'empress'])
pretty_print_scores_en(['pretty', 'handsome'])
pretty_print_scores_en(['programmer', 'engineer', 'scientist', 'artist', 'doctor', 'nurse'])
pretty_print_scores_en(['chef', 'cook', 'homemaker', 'dishwasher'])
pretty_print_scores_en(['car', 'bike', 'football', 'truck', 'driver'])
pretty_print_scores_en(['stupid', 'dumb', 'ditzy', 'airhead', 'scatterbrained'])


===== baseline words =====
M: man: 0.08796428889036179; male: -0.20079752802848816; boy: -0.04156001657247543; father: 0.12683464586734772
F: woman: -0.40482816100120544; female: -0.33166825771331787; girl: -0.3568439185619354; mother: -0.308342307806015
===== test words =====
leader: 0.18232303857803345; authority: 0.09276363253593445; president: 0.1274125725030899; king: 0.27469533681869507; queen: -0.16571253538131714; premier: 0.15453550219535828; emperor: 0.17684833705425262; empress: -0.21059739589691162
pretty: -0.028876882046461105; handsome: 0.0371420681476593
programmer: 0.1263071745634079; engineer: 0.1671569049358368; scientist: 0.07804803550243378; artist: -0.037671640515327454; doctor: -0.06677766889333725; nurse: -0.32890164852142334
chef: -0.025211520493030548; cook: -0.07091394066810608; homemaker: -0.34615224599838257; dishwasher: -0.12109534442424774
car: 0.10213074088096619; bike: 0.0933547168970108; football: 0.19292551279067993; truck: 0.03709475323557854; driver: